<a href="https://colab.research.google.com/github/diseasemodeling/MIE_525_625/blob/main/Lectures/4_MLP_heads/4e_Pipelines_HuggingFace.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Note:
This notebook contains text/code directly copied from cited sources for purpose of class lecture only.
# NOTE: Hugging Face Access & Tokens
* Public Open-Source Models (No Token Required): Most standard open-source models can be downloaded anonymously. You do not need to create a Hugging Face account or provide any API keys for your code to work.
  * Example: pipeline("text-generation") defaults to GPT-2.
  * Models in this notebook belown to this category.
  * Any errors may relate to changes in huggingface on how models are acccesed. Lookup orginal source for changes Source: https://huggingface.co/learn/llm-course/chapter1/3 Hugging Face, The Hugging Face Course, 2022, https://huggingface.co/course, 2022
  * Other fully public models include Mistral-7B-v0.1 (mistralai/Mistral-7B-v0.1) and Microsoft's Phi-3 (microsoft/Phi-3-mini-4k-instruct).
  
* Gated Open-Source Models (Free Token Required): Some models are open-source but require you to accept a user license agreement on the Hugging Face website before downloading.
  * Creating an account and generating this token is completely free, but the code will crash with a 401 Error if the token is missing.
  * Example: Meta's Llama 3 (meta-llama/Meta-Llama-3-8B) and Google's Gemma (google/gemma-7b) both require you to accept their terms on their Hugging Face model page and paste your free token into your environment.


# Using Transformers in HuggingFace
Source: https://huggingface.co/learn/llm-course/chapter1/3
Hugging Face, The Hugging Face Course, 2022, https://huggingface.co/course, 2022

What is HugggingFace Transformers? *It is an open-source Python library that provides access to thousands of pre-trained Transformers models for natural language processing (NLP), computer vision, audio tasks, and more.*

- *The most basic object in the 🤗 Transformers library is the pipeline() function.*
- *It connects a model with its necessary preprocessing and postprocessing steps, allowing us to directly input any text and get an intelligible answer*
- **Pipeline selects a particular pretrained model that has been fine-tuned for specific tasks**

[**"Awesome projects built with Transformers"**](https://github.com/huggingface/transformers/blob/main/awesome-transformers.md)

# **Pipelines in HuggingFace - Use pretrained models that have been fine-tuned for specific tasks**
Text pipelines

    text-generation: Generate text from a prompt
    text-classification: Classify text into predefined categories
    summarization: Create a shorter version of a text while preserving key information
    translation: Translate text from one language to another
    zero-shot-classification: Classify text without prior training on specific labels
    feature-extraction: Extract vector representations of text

Image pipelines

    image-to-text: Generate text descriptions of images
    image-classification: Identify objects in an image
    object-detection: Locate and identify objects in images

Audio pipelines

    automatic-speech-recognition: Convert speech to text
    audio-classification: Classify audio into categories
    text-to-speech: Convert text to spoken audio

Multimodal pipelines

    image-text-to-text: Respond to an image based on a text prompt

Best suited modes

  ### 🔧 NLP Tasks and Suitable Model Architectures

| **Task** | **Suggested Architecture** | **Examples** |
|---------|----------------------------|--------------|
| Text classification (sentiment, topic) | Encoder | BERT, RoBERTa |
| Text generation (creative writing) | Decoder | GPT, LLaMA |
| Translation | Encoder–Decoder | T5, BART |
| Summarization | Encoder–Decoder | BART, T5 |
| Named entity recognition | Encoder | BERT, RoBERTa |
| Question answering (extractive) | Encoder | BERT, RoBERTa |
| Question answering (generative) | Encoder–Decoder or Decoder | T5, GPT |
| Conversational AI | Decoder | GPT, LLaMA |


    

# **EXAMPLE 1: USE MODEL FOR SENTIMENT ANALYSIS**
Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

## **DIRECTLY USE PIPELINE**

In [ ]:
!pip install -q datasets evaluate transformers[sentencepiece]

In [ ]:
from transformers import pipeline

classifier = pipeline("sentiment-analysis")#initializes a pre-trained sentiment analysis model
classifier("I've been waiting for a HuggingFace course my whole life.")#classify user-defied sentence



[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[{'label': 'POSITIVE', 'score': 0.9598048329353333}]

In [ ]:
classifier(
    ["I've been waiting for a HuggingFace course my whole life.", "I hate this so much!"]
)

[{'label': 'POSITIVE', 'score': 0.9598048329353333},
 {'label': 'NEGATIVE', 'score': 0.9994558691978455}]

## **SEE WHATS HAPPENING BEHIND THE PIPELINE**

Above pipeline groups together three steps: preprocessing, passing the inputs through the model, and postprocessing:
![](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/full_nlp_pipeline.svg)
![](https://huggingface.co/datasets/huggingface-course/documentation-images/resolve/main/en/chapter2/transformer_and_head.svg)

In [ ]:
from transformers import AutoTokenizer #automatically use the correct tokenizer specific to model
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"# DistilBERT fine-tuned for sentiment analysis (using SST-2 dataset https://huggingface.co/datasets/stanfordnlp/sst2) (There are over 1M+ Transformers model checkpoints on the Hugging Face Hub we can use.)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
raw_inputs = [
    "I've been waiting for a HuggingFace course my whole life.",
    "I hate this so much!",
]
inputs = tokenizer(raw_inputs, padding=True, truncation=True, return_tensors="pt")
print("Print tokenized inputs: input ids + attention mask \n", inputs)
#-------------------

from transformers import AutoModel
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModel.from_pretrained(checkpoint) #AutoModel loads only the base transformer encoder, without any classification head.
outputs = model(**inputs)
print("Print last_hidden_state: shape (batch_size, sequence_length, hidden_dim) \n", outputs.last_hidden_state.shape) #DistilBERT, hidden_dim = 768
#------------------

from transformers import AutoModelForSequenceClassification
checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint) # This loads the DistilBERT encoder plus a classifier layer trained for SST-2 (positive/negative). Output includes: logits: raw predictions for each class.
outputs = model(**inputs)
print(outputs.logits.shape)
print("Print logits: raw predictions for each class \n", outputs.logits)

import torch
predictions = torch.nn.functional.softmax(outputs.logits, dim=-1) #Softmax converts logits into probabilities.
print("Print predictions:\n", predictions)
model.config.id2label #Returns the mapping from class index to label name.

Print tokenized inputs: input ids + attention mask 
 {'input_ids': tensor([[  101,  1045,  1005,  2310,  2042,  3403,  2005,  1037, 17662, 12172,
          2607,  2026,  2878,  2166,  1012,   102],
        [  101,  1045,  5223,  2023,  2061,  2172,   999,   102,     0,     0,
             0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0]])}


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.bias       | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Print last_hidden_state: shape (batch_size, sequence_length, hidden_dim) 
 torch.Size([2, 16, 768])


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

torch.Size([2, 2])
Print logits: raw predictions for each class 
 tensor([[-1.5607,  1.6123],
        [ 4.1692, -3.3464]], grad_fn=<AddmmBackward0>)
Print predictions:
 tensor([[4.0195e-02, 9.5980e-01],
        [9.9946e-01, 5.4418e-04]], grad_fn=<SoftmaxBackward0>)


{0: 'NEGATIVE', 1: 'POSITIVE'}

# **OTHER PIPELINE EXAMPLES**

In [ ]:
#from transformers import pipeline

classifier = pipeline("zero-shot-classification")
classifier(
    "This is a course about the Transformers library",
    candidate_labels=["education", "politics", "business"],
)

[transformers] No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

{'sequence': 'This is a course about the Transformers library',
 'labels': ['education', 'business', 'politics'],
 'scores': [0.8445985913276672, 0.11197450757026672, 0.04342687129974365]}

In [ ]:
#from transformers import pipeline

generator = pipeline("text-generation")
generator("In this course, we will teach you how to")

[transformers] No model was supplied, defaulted to HuggingFaceTB/SmolLM3-3B and revision a07cc9a.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': "In this course, we will teach you how to create your own 2D platformer game using Python and the Pygame library. \n\nTo get started, you will need to install Python and Pygame. You can download Python from the official Python website and Pygame from the Pygame website.\n\nOnce you have installed Python and Pygame, you can create a new file and start coding. The first thing you will need to do is import the Pygame library and initialize the game.\n\nHere is a basic outline of the steps you will need to take to create your game:\n\n1. Import the Pygame library and initialize the game.\n2. Create the game world and set up the game loop.\n3. Define the player character and set up the player's movement.\n4. Define the game objects and set up the game logic.\n5. Draw the game world and the player character on the screen.\n6. Update the game state in the game loop.\n7. Handle player input and update the game state accordingly.\n8. Render the game state and display it on t

In [ ]:
#from transformers import pipeline

generator = pipeline("text-generation", model="distilgpt2")
generator(
    "In this course, we will teach you how to",
    max_length=30,
    num_return_sequences=2,
)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'num_return_sequences', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=30) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': 'In this course, we will teach you how to use the techniques used to take advantage of the strengths and weaknesses of the material.'},
 {'generated_text': 'In this course, we will teach you how to understand the problem in your own hands and from an intuitive perspective.'}]

In [ ]:
#from transformers import pipeline
num_options=1
generator = pipeline("text-generation", model="gpt2")
prompt = f"""
        Generate {num_options} realistic meals.
        For each meal, output:
        name, nutrition_score (1-10), cost (1-10), prep_time (1-10).
        Return ONLY a Python list of lists in this format:
        [[nutrition, cost, time], ...]
        """
output = generator(prompt, max_length=150, do_sample=True, temperature=0.7)[0]["generated_text"]
#output = generator(prompt, max_length=150)
print(output)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'temperature', 'do_sample', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=256) and `max_length`(=150) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



        Generate 1 realistic meals.
        For each meal, output:
        name, nutrition_score (1-10), cost (1-10), prep_time (1-10).
        Return ONLY a Python list of lists in this format:
        [[nutrition, cost, time], ...]
                            

[size_start_time, size_end_time]

#

# A list of food items:

#

# - Name: This list will contain food items that are not in the default list.

# - Cost: This list will contain only the cost of food items.

# - Prep: This list will contain the prep and time to prep the food items.

# - Time: This list will serve as the time to prep the food items.

# - Total: This list will serve as the total time to prep the food items.

#

# - The food item will be used only if it is the same size as the rest of the list.

#

# - In the case of food items with no items, the food item will be used only if they are of the same size.

# - In the case of items with two or more items, the food item will be used only if its size is larger than th

In [ ]:
#from transformers import pipeline

unmasker = pipeline("fill-mask")
unmasker("This course will teach you all about <mask> models.", top_k=2)

[transformers] No model was supplied, defaulted to distilbert/distilroberta-base and revision fb53ab8.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

[transformers] RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'score': 0.19619713723659515,
  'token': 30412,
  'token_str': ' mathematical',
  'sequence': 'This course will teach you all about mathematical models.'},
 {'score': 0.040527280420064926,
  'token': 38163,
  'token_str': ' computational',
  'sequence': 'This course will teach you all about computational models.'}]

In [ ]:
#from transformers import pipeline

ner = pipeline("ner", aggregation_strategy="simple")
ner("My name is Sylvain and I work at Hugging Face in Brooklyn.")

[transformers] No model was supplied, defaulted to dbmdz/bert-large-cased-finetuned-conll03-english and revision 4c53496.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dbmdz/bert-large-cased-finetuned-conll03-english
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.weight | UNEXPECTED |  | 
bert.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'entity_group': 'PER',
  'score': np.float32(0.9981694),
  'word': 'Sylvain',
  'start': 11,
  'end': 18},
 {'entity_group': 'ORG',
  'score': np.float32(0.9796019),
  'word': 'Hugging Face',
  'start': 33,
  'end': 45},
 {'entity_group': 'LOC',
  'score': np.float32(0.9932106),
  'word': 'Brooklyn',
  'start': 49,
  'end': 57}]

In [ ]:
#from transformers import pipeline
'''
question_answerer = pipeline(
    "question-answering",
    model="distilbert-base-cased-distilled-squad"
)
question_answerer(
    question="Where do I work?",
    context="My name is Sylvain and I work at Hugging Face in Brooklyn",
)
'''

'\nquestion_answerer = pipeline(\n    "question-answering", \n    model="distilbert-base-cased-distilled-squad"\n)\nquestion_answerer(\n    question="Where do I work?",\n    context="My name is Sylvain and I work at Hugging Face in Brooklyn",\n)\n'

In [ ]:
#from transformers import pipeline
'''
summarizer = pipeline("summarization")
summarizer(
    """
    America has changed dramatically during recent years. Not only has the number of
    graduates in traditional engineering disciplines such as mechanical, civil,
    electrical, chemical, and aeronautical engineering declined, but in most of
    the premier American universities engineering curricula now concentrate on
    and encourage largely the study of engineering science. As a result, there
    are declining offerings in engineering subjects dealing with infrastructure,
    the environment, and related issues, and greater concentration on high
    technology subjects, largely supporting increasingly complex scientific
    developments. While the latter is important, it should not be at the expense
    of more traditional engineering.

    Rapidly developing economies such as China and India, as well as other
    industrial countries in Europe and Asia, continue to encourage and advance
    the teaching of engineering. Both China and India, respectively, graduate
    six and eight times as many traditional engineers as does the United States.
    Other industrial countries at minimum maintain their output, while America
    suffers an increasingly serious decline in the number of engineering graduates
    and a lack of well-educated engineers.
"""
)
'''

'\nsummarizer = pipeline("summarization")\nsummarizer(\n    """\n    America has changed dramatically during recent years. Not only has the number of\n    graduates in traditional engineering disciplines such as mechanical, civil,\n    electrical, chemical, and aeronautical engineering declined, but in most of\n    the premier American universities engineering curricula now concentrate on\n    and encourage largely the study of engineering science. As a result, there\n    are declining offerings in engineering subjects dealing with infrastructure,\n    the environment, and related issues, and greater concentration on high\n    technology subjects, largely supporting increasingly complex scientific\n    developments. While the latter is important, it should not be at the expense\n    of more traditional engineering.\n\n    Rapidly developing economies such as China and India, as well as other\n    industrial countries in Europe and Asia, continue to encourage and advance\n    the teachi

In [ ]:
#from transformers import pipeline
'''
translator = pipeline("translation", model="Helsinki-NLP/opus-mt-fr-en")
translator("Ce cours est produit par Hugging Face.")
'''

'\ntranslator = pipeline("translation", model="Helsinki-NLP/opus-mt-fr-en")\ntranslator("Ce cours est produit par Hugging Face.")\n'

In [ ]:
#from transformers import pipeline

image_classifier = pipeline(
    task="image-classification", model="google/vit-base-patch16-224"
)
result = image_classifier(
    "https://diseasemodeling.github.io/docs/assets/LabOverview.png"
)
print(result)

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

[{'label': 'web site, website, internet site, site', 'score': 0.6211074590682983}, {'label': 'scoreboard', 'score': 0.10771726816892624}, {'label': 'slot, one-armed bandit', 'score': 0.06261196732521057}, {'label': 'analog clock', 'score': 0.01442665047943592}, {'label': 'crossword puzzle, crossword', 'score': 0.013500436209142208}]


In [ ]:
#from transformers import pipeline


from transformers import pipeline
segmenter = pipeline("image-segmentation", model="facebook/detr-resnet-50-panoptic")

segments = segmenter("https://huggingface.co/datasets/Narsil/image_dummy/raw/main/parrots.png")



Loading weights:   0%|          | 0/562 [00:00<?, ?it/s]

**display actual and segmented image**

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO

# URL of the original image used for segmentation
image_url = "https://huggingface.co/datasets/Narsil/image_dummy/raw/main/parrots.png"

# Fetch and open the original image
response = requests.get(image_url)
original_image = Image.open(BytesIO(response.content))

# Create a figure with subplots: one for the original image and one for each mask
num_segments = len(segments)
fig, axes = plt.subplots(1, num_segments + 1, figsize=(5 * (num_segments + 1), 5))

# Display the original image in the first subplot
axes[0].imshow(original_image)
axes[0].set_title("Original Image")
axes[0].axis("off")

# Display each segmented mask in subsequent subplots
for i, segment in enumerate(segments):
    ax = axes[i + 1]
    ax.imshow(segment['mask'], cmap='gray') # Masks are typically grayscale
    ax.set_title(f"Segment {i + 1}: {segment['label']}")
    ax.axis("off")

plt.tight_layout()
plt.show()